```bash
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[OptimalTransport]'
```

- Introduction (/3)
- Preparation et analyse des données (/3)
- Application des méthodes de pre processing (/5)
- Application des méthodes de post processing (/5)
- Analyse, compréhension (/3)
- Conclusion (/1)

## Introduction

nous disposons désormais des images elles-mêmes, ce qui nous permet d'entraîner un véritable modèle de prédiction. L’objectif est de construire un pipeline complet comportant :
- un prétraitement visant à atténuer les biais avant l'entraînement (par exemple via l'algorithme LFR),
- un modèle de classification basé sur les images pour prédire les maladies,
- un post-traitement appliqué aux prédictions pour corriger d’éventuelles inégalités restantes (via l'algorithme Equalized Odds Postprocessing par exemple).

Ce rapport présente la mise en œuvre de ce pipeline, les défis rencontrés, et une évaluation de l'impact des différentes étapes sur la qualité et l’équité des prédictions.

In [3]:
import utils
import os
import pandas as pd
from constants import *
from train_classifieur import train_classifier, pred_classifier
from aif360.datasets import BinaryLabelDataset
import plotly.express as px


utils.load_env_file()
data_dir = os.getenv("DATA_DIR", "data/default/")
og_metadata_filename="original_metadata.csv"
og_metadata_path = data_dir + og_metadata_filename
print("Travaille sur : ", data_dir)
print(og_metadata_path)

Travaille sur :  ./data/SAILLANT_ARTHUR/selected_data/
./data/SAILLANT_ARTHUR/selected_data/original_metadata.csv


In [4]:
# variables et fonctions importante 

map_genre = {"M": 0, "F": 1}
map_viewposition = {"AP": 0, "PA": 1}
map_pred = {"sain": 0, "malade": 1}

fav_lbl = map_pred["sain"]
unfav_lbl = map_pred["malade"]
protected_attributes = ['Patient Gender', '+40ans']

protected_attribute = protected_attributes[1]


In [5]:

def convert_to_all_numerical(df):
    # Define paths to the train repository
    train_sain_path = data_dir+"train/sain"
    train_malade_path = data_dir+"train/malade"

    # Get the list of image filenames in the train repository
    train_images = set(os.listdir(train_sain_path) + os.listdir(train_malade_path))

    df.columns = df.columns.str.strip()
    if "in_train" not in df.columns:
        df["in_train"] = df["Image Index"].apply(lambda x: 1 if x in train_images else 0)
    if 'Finding Labels' in df.columns:
        df_ohe = df['Finding Labels'].str.get_dummies(sep='|').astype(bool)
        df = df.drop(columns=['Finding Labels']).join(df_ohe)
    if "preds" in df.columns and not pd.api.types.is_numeric_dtype(df["preds"]):
        df["preds"] = df["preds"].map({"sain": 0, "malade": 1})
    if "labels" in df.columns and not pd.api.types.is_numeric_dtype(df["labels"]):
        df["labels"] = df["labels"].map({"sain": 0, "malade": 1})
    if not pd.api.types.is_numeric_dtype(df["Patient Gender"]):
        df["Patient Gender"] = df["Patient Gender"].map(map_genre)
    if "View Position" in df.columns and not pd.api.types.is_numeric_dtype(df["View Position"]):
        df["View Position"] = df["View Position"].map(map_viewposition)
    if "+40ans" not in df.columns:
        df["+40ans"] = (df["Patient Age"] > 40).astype(int) 
    return df

In [6]:

from aif360.sklearn.metrics import *


def get_group_metrics(
    y_true,
    y_pred=None,
    prot_attr=None,
    priv_group=1,
    pos_label=1,
    sample_weight=None,
):
    group_metrics = {}
    group_metrics["base rate"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["SPD"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["DI"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    if not y_pred is None:
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["smoothed_edf"] = smoothed_edf(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
    return group_metrics


2025-04-06 10:18:21.169487: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743927501.188827   85221 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743927501.194063   85221 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743927501.206975   85221 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743927501.206998   85221 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743927501.207000   85221 computation_placer.cc:177] computation placer alr

In [7]:
def train_and_predict(metadata_csv, outputcsv, force_training=False):
    logdir = "./expe_log/"
    os.makedirs(logdir, exist_ok=True)
    csv_out = os.path.join(logdir, outputcsv)

    if force_training or not os.path.exists(csv_out):
        print("Entrainement du classifieur...")
        ckpt_path, ckpt_score = train_classifier(
            logdir=logdir,
            datadir=data_dir,
            csv=data_dir+metadata_csv,
        )
        print("Génerations des predictions...")
        pred_classifier(
            datadir=data_dir,
            csv_in=data_dir+metadata_csv,
            csv_out=csv_out,
            ckpt_path=ckpt_path
        )
    else:
        print(f"Les prédiction existent déjà à {csv_out} -- abandon de l'entraînement")

def intoBinaryLabelDataset(df):
    manquantes = [attribute for attribute in protected_attributes if attribute not in df.columns]

    if manquantes:
        raise ValueError(f"Les colonnes protégées suivantes sont manquantes dans le dataset : {', '.join(manquantes)}")

    dataset = BinaryLabelDataset(
        favorable_label=fav_lbl,  # "Sain" est la classe favorable
        unfavorable_label=unfav_lbl,  # "Malade" est la classe défavorable
        df=df,
        label_names=["labels"],
        protected_attribute_names=protected_attributes
    )
    return dataset

def getMetric(df, prot_attr):
    if isinstance(prot_attr, list) :
        raise RuntimeError("On ne peut pas faire de metriquesurplusieur attr protegé")
    df = convert_to_all_numerical(df)
    preds = df["preds"]
    labels= df["labels"]
    weights = df["WEIGHTS"]

    metrics_after_reweight = get_group_metrics(
        y_true=labels,
        y_pred=preds,
        prot_attr=df[prot_attr],
        priv_group=1,
        pos_label=1,
        sample_weight=weights
    )
    return metrics_after_reweight

    


## Préparations des données

Notamment pour les converitir dans un ``BinaryLabelDataset``

In [8]:
df = pd.read_csv(og_metadata_path)

print(df.columns)
imageid_df = df.copy()[["Image Index", patientid]]
original_df = df.copy()
df = convert_to_all_numerical(df)

df.head() # Y'a toujours Image index !!

Index(['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID',
       'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width',
       'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'WEIGHTS'],
      dtype='object')


,Image Index,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],...,Fibrosis,Hernia,Infiltration,Mass,No Finding,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,+40ans
0,00000042_006.png,6,42,71,0,0,3056,2544,0.139000,0.139000,...,False,False,True,False,False,False,False,False,False,1
1,00000048_000.png,0,48,46,1,1,2834,2641,0.143000,0.143000,...,False,False,False,False,True,False,False,False,False,1
2,00000096_000.png,0,96,67,1,1,2646,2829,0.143000,0.143000,...,False,False,False,False,False,False,False,False,False,1
3,00000097_000.png,0,97,83,1,1,2021,1865,0.194311,0.194311,...,False,False,False,False,True,False,False,False,False,1
4,00000100_000.png,0,100,60,0,1,2500,2048,0.171000,0.171000,...,False,False,True,False,False,False,False,False,False,1


In [9]:
# recupperer les predictions sans aucun changement
train_and_predict(og_metadata_filename, "original_preds.csv")

Les prédiction existent déjà à ./expe_log/original_preds.csv -- abandon de l'entraînement


In [10]:
preddf = pd.read_csv("./expe_log/original_preds.csv")
preddf = convert_to_all_numerical(preddf)

metrics_before_training = getMetric(preddf, protected_attribute)

def compare_to_base_preds(metrics_after):
    for metric in metrics_before_training.keys():
        before = metrics_before_training[metric]
        after = metrics_after[metric]
        change = after - before
        print(f"{before:.4f} ---- {metric} ---> {after:.4f}, (diff = {change:.4f})")


## Analyse

In [11]:
from sklearn.metrics import confusion_matrix
def plot_confusion_matrix(y_true, y_pred, labels=["sain", "malade"], normalize=False, title="Matrice de Confusion"):
    cm = confusion_matrix(y_true, y_pred)
    
    if normalize:
        cm = cm.astype('float') / len(y_true) * 100
    
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    
    fig = px.imshow(cm_df, 
                    labels=dict(x="Prédiction", y="Vérité", color="Fréquence (%)" if normalize else "Fréquence"), 
                    x=labels, 
                    y=labels, 
                    color_continuous_scale='Blues',
                    range_color=[0, 100] if normalize else None) 
    
    for i in range(len(cm_df)):
        for j in range(len(cm_df.columns)):
            fig.add_annotation(
                x=j,
                y=i,
                text=f'{cm_df.iloc[i, j]:.2f}%' if normalize else f'{cm_df.iloc[i, j]}',
                showarrow=False,
                font=dict(color="black", size=14),
                align="center"
            )
    
    fig.update_layout(title=title, xaxis_title="Prédiction", yaxis_title="Vérité")
    fig.show()

def plot_confusion_matrix_by_group(y_true, y_pred, df, group_columns, labels=None, normalize=False):
    """
    Affiche des matrices de confusion séparées pour chaque groupe défini par group_columns.
    """
    for group_value, group_df in df.groupby(group_columns):
        y_true_group = y_true[group_df.index]
        y_pred_group = y_pred[group_df.index]
        
        print(f"Matrice de confusion pour {group_columns}: {group_value}")
        plot_confusion_matrix(y_true_group, y_pred_group, labels, normalize, title=f"Matrice de Confusion ({group_columns}={group_value})")


In [12]:
preddf['+40ans'] = preddf['Patient Age'] >= 40
plot_confusion_matrix_by_group(preddf["labels"], preddf["preds"], preddf, group_columns=["+40ans"], labels=["sain", "malade"])

Matrice de confusion pour ['+40ans']: (False,)


Matrice de confusion pour ['+40ans']: (True,)


In [16]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix

# Initialisation du DataFrame global pour les taux d'erreur
error_rate_df = pd.DataFrame(columns=['method', 'global', '+40ans', '-40ans', 'M', 'F'])

def add_error_rate(df, method):
    global error_rate_df  # Accéder au DataFrame global

    # Calcul du taux d'erreur global
    y_true = df["labels"].values
    y_pred = df["preds"].values
    cm = confusion_matrix(y_true, y_pred)
    total = cm.sum()
    correct = np.trace(cm)
    global_error_rate = (total - correct) / total * 100
    
    df['age_binary'] = (df['Patient Age'] > 40).astype(int)
    age_error_rate = df.groupby('age_binary').apply(
        lambda group_df: (confusion_matrix(group_df["labels"].values, group_df["preds"].values).sum() -
                          np.trace(confusion_matrix(group_df["labels"].values, group_df["preds"].values))) /
                         confusion_matrix(group_df["labels"].values, group_df["preds"].values).sum() * 100
    )
    sex_error_rate = df.groupby('Patient Gender').apply(
        lambda group_df: (confusion_matrix(group_df["labels"].values, group_df["preds"].values).sum() -
                          np.trace(confusion_matrix(group_df["labels"].values, group_df["preds"].values))) /
                         confusion_matrix(group_df["labels"].values, group_df["preds"].values).sum() * 100
    )
    new_row = pd.DataFrame({
        'method': [method],
        'global': [global_error_rate],
        '+40ans': [age_error_rate.get(1, np.nan)],  # Age > 40
        '-40ans': [age_error_rate.get(0, np.nan)],  # Age <= 40
        'M': [sex_error_rate.get(1, np.nan)],       # Sexe = M (homme)
        'F': [sex_error_rate.get(0, np.nan)]        # Sexe = F (femme)
    })

    error_rate_df = pd.concat([error_rate_df, new_row], ignore_index=True)
    



In [17]:
add_error_rate(preddf, 'Normal')
error_rate_df

/tmp/ipykernel_85221/3799069778.py:20: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_85221/3799069778.py:25: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_85221/3799069778.py:39: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result 

,method,global,+40ans,-40ans,M,F
0,Normal,23.8,25.255624,21.072797,24.25107,23.404255


## pre processing

#### Pre pre processing

In [18]:

og_preddf = pd.read_csv("expe_log/original_preds.csv")
og_preddf = convert_to_all_numerical(og_preddf)




filtered_df = og_preddf.drop(["View Position", "Finding Labels", "Image Index"], axis=1, errors="ignore")
train_df = filtered_df[filtered_df["in_train"]==1].copy().reset_index()
test_df = filtered_df[filtered_df["in_train"]==0].copy().reset_index()

dataset = intoBinaryLabelDataset(filtered_df)
train_dataset = intoBinaryLabelDataset(train_df)
test_dataset = intoBinaryLabelDataset(test_df)


a=len(train_dataset.instance_weights)
b=len(test_dataset.instance_weights)
c=len(dataset.instance_weights)
assert(a+b==c)
train_df

,index,Follow-up #,Patient ID,Patient Age,Patient Gender,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],WEIGHTS,...,Fibrosis,Hernia,Infiltration,Mass,No Finding,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,+40ans
0,4,0,100,60,0,2500,2048,0.171000,0.171000,1,...,False,False,True,False,False,False,False,False,False,1
1,7,0,202,66,0,3056,2544,0.139000,0.139000,1,...,False,False,False,False,True,False,False,False,False,1
2,8,0,204,37,0,2674,2878,0.143000,0.143000,1,...,True,False,False,False,False,False,False,False,False,0
3,9,0,208,40,0,2048,2500,0.171000,0.171000,1,...,False,False,False,False,False,False,True,False,False,0
4,10,0,219,78,0,2500,2048,0.171000,0.171000,1,...,False,False,False,False,False,False,False,False,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1120,1495,0,30775,20,0,2021,2021,0.194311,0.194311,1,...,False,False,False,False,True,False,False,False,False,0
1121,1496,0,30783,57,0,2021,2021,0.194311,0.194311,1,...,False,False,False,False,True,False,False,False,False,1
1122,1497,0,30788,61,1,2021,2021,0.194311,0.194311,1,...,False,False,False,False,True,False,False,False,False,1
1123,1498,0,30792,10,1,1775,1712,0.194311,0.194311,1,...,False,False,False,False,True,False,False,False,False,0


#### reweight

In [19]:
sensitive_attr = "+40ans"
unprivileged_groups, privileged_groups=[{sensitive_attr: 0}], [{sensitive_attr: 1}]

In [20]:
from aif360.algorithms.preprocessing import Reweighing
from train_classifieur import train_classifier, pred_classifier



rw = Reweighing(unprivileged_groups, privileged_groups)
rw.fit(train_dataset)
transformed_dataset = rw.transform(dataset)

csv_df = original_df.copy()
csv_df["WEIGHTS"] = transformed_dataset.instance_weights
csv_df.to_csv(data_dir+"reweighted_metadata.csv", index=False)


In [21]:
train_and_predict("reweighted_metadata.csv", "reweighted_preds.csv")

Les prédiction existent déjà à ./expe_log/reweighted_preds.csv -- abandon de l'entraînement


In [22]:
rw_pred = pd.read_csv("./expe_log/reweighted_preds.csv")
rw_pred = convert_to_all_numerical(rw_pred)
metrics_after_reweight = getMetric(rw_pred, sensitive_attr)

In [23]:
compare_to_base_preds(metrics_after_reweight)

0.4573 ---- base rate ---> 0.4573, (diff = -0.0000)
-0.2070 ---- SPD ---> 0.0068, (diff = 0.2138)
0.5608 ---- DI ---> 1.0186, (diff = 0.4578)
-0.1404 ---- equal_opportunity_difference ---> 0.0206, (diff = 0.1610)
-0.1347 ---- average_odds_difference ---> 0.0113, (diff = 0.1459)
-0.0595 ---- conditional_demographic_disparity ---> 0.0020, (diff = 0.0615)
0.5767 ---- smoothed_edf ---> 0.0187, (diff = -0.5579)
0.2311 ---- df_bias_amplification ---> 0.0006, (diff = -0.2305)


In [24]:
add_error_rate(rw_pred, "Rewight")
error_rate_df

/tmp/ipykernel_85221/3799069778.py:20: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_85221/3799069778.py:25: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,method,global,+40ans,-40ans,M,F
0,Normal,23.8,25.255624,21.072797,24.25107,23.404255
1,Rewight,27.4,28.834356,24.712644,26.81883,27.909887


In [25]:

plot_confusion_matrix_by_group(rw_pred["labels"], rw_pred["preds"], rw_pred, group_columns=["+40ans"], labels=[0, 1])

Matrice de confusion pour ['+40ans']: (0,)


Matrice de confusion pour ['+40ans']: (1,)


#### DIR

In [26]:
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import DisparateImpactRemover
import pandas as pd

def apply_disparate_impact_remover(original_df, repair_level=1.0):
    label_col = "labels"

    protected_attr = "+40ans"

    # Colonnes à garder pour la réparation
    dir_features = ["Patient Age", "Patient Gender", protected_attr, label_col, "WEIGHTS"] 
    
    df_dir = original_df[dir_features].copy()
    dataset = BinaryLabelDataset(
        df=df_dir,
        label_names=[label_col],
        protected_attribute_names=[protected_attr]
    )

    # pour les restaurer ensuite
    patient_ids = original_df["Patient ID"].astype(str).tolist()
    dataset.instance_names = [[pid] for pid in patient_ids]

    dir = DisparateImpactRemover(sensitive_attribute=protected_attr, repair_level=repair_level)
    repaired_dataset = dir.fit_transform(dataset)
    repaired_df = pd.DataFrame(
        data=repaired_dataset.features,
        columns=repaired_dataset.feature_names
    )
    repaired_df[label_col] = repaired_dataset.labels
    # on remet les ids et les images
    repaired_df["Patient ID"] = [int(pid[0]) for pid in repaired_dataset.instance_names]
    imageid_df["Patient ID"] = imageid_df["Patient ID"].astype(int)
    repaired_df = repaired_df.merge(imageid_df, on="Patient ID", how="left")

    columns_to_add = ["in_train"]
    for col in columns_to_add:
        repaired_df[col] = original_df[col].values

    repaired_df["+40ans"] = (repaired_df["Patient Age"] >= 40).astype(int)
    return repaired_df


In [27]:
# pournepas utiliser d'info du datatest dans le train -> data leakage

repaired_train_df = apply_disparate_impact_remover(train_df)
repaired_test_df =  apply_disparate_impact_remover(test_df)

repaired_df = pd.concat([repaired_train_df, repaired_test_df], ignore_index=True)

repaired_df.to_csv(data_dir+"dir_metadata.csv", index=False)


In [28]:
train_and_predict("dir_metadata.csv", "dir_preds.csv")

Les prédiction existent déjà à ./expe_log/dir_preds.csv -- abandon de l'entraînement


In [29]:
dir_df = pd.read_csv("./expe_log/dir_preds.csv")
dir_df = convert_to_all_numerical(dir_df)
metrics_after_dir = getMetric(dir_df, sensitive_attr)
compare_to_base_preds(metrics_after_dir)

0.4573 ---- base rate ---> 0.4573, (diff = 0.0000)
-0.2070 ---- SPD ---> 0.0383, (diff = 0.2453)
0.5608 ---- DI ---> 1.1150, (diff = 0.5541)
-0.1404 ---- equal_opportunity_difference ---> 0.1697, (diff = 0.3101)
-0.1347 ---- average_odds_difference ---> 0.0457, (diff = 0.1804)
-0.0595 ---- conditional_demographic_disparity ---> -0.0039, (diff = 0.0556)
0.5767 ---- smoothed_edf ---> 0.0967, (diff = -0.4800)
0.2311 ---- df_bias_amplification ---> 0.0510, (diff = -0.1801)


In [30]:
add_error_rate(dir_df, "Dir")
error_rate_df

/tmp/ipykernel_85221/3799069778.py:20: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_85221/3799069778.py:25: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,method,global,+40ans,-40ans,M,F
0,Normal,23.8,25.255624,21.072797,24.251070,23.404255
1,Rewight,27.4,28.834356,24.712644,26.818830,27.909887
2,Dir,30.0,NaN,30.000000,28.815977,31.038798


In [31]:
plot_confusion_matrix_by_group(dir_df["labels"], dir_df["preds"], dir_df, group_columns=["+40ans"], labels=[0, 1])

Matrice de confusion pour ['+40ans']: (0,)


Matrice de confusion pour ['+40ans']: (1,)


#### LFR

In [32]:
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import LFR
import pandas as pd

def apply_lfr(df, maxiter=5000, maxfun=5000):


    label_col = "labels"

    protected_attr = "+40ans"

    # ✅ Colonnes à garder pour la réparation (features + protected + label)
    dir_features = ["Patient Age", "Patient Gender", protected_attr, label_col, "WEIGHTS"] 

    # 1. Création du dataset minimal pour AIF360
    df_dir = df[dir_features].copy()

    
    dataset = BinaryLabelDataset(
        df=df_dir,
        label_names=[label_col],
        protected_attribute_names=[protected_attr]
    )

    # 2. Stockage des Patient ID pour les restaurer ensuite
    patient_ids = df["Patient ID"].astype(str).tolist()
    dataset.instance_names = [[pid] for pid in patient_ids]

    # 3. Application de LFR
    TR = LFR(
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
        k=5,  # Paramètre d'équilibrage des représentations
        Ax=0.001, Ay=0.1, Az=1.0,
        print_interval=500,
        verbose=1,
        seed=None
    )

    TR = TR.fit(dataset, maxiter=maxiter, maxfun=maxfun)
    repaired_dataset = TR.transform(dataset)

    # 4. Reconstruction du DataFrame réparé avec les bonnes colonnes
    repaired_df = pd.DataFrame(
        data=repaired_dataset.features,
        columns=repaired_dataset.feature_names
    )
    repaired_df[label_col] = repaired_dataset.labels

    # 5. Réinsertion des Patient ID
    repaired_df["Patient ID"] = [int(pid[0]) for pid in repaired_dataset.instance_names]

    # 6. Fusion avec imageid_df pour ajouter les chemins d’image
    imageid_df["Patient ID"] = imageid_df["Patient ID"].astype(int)
    repaired_df = repaired_df.merge(imageid_df, on="Patient ID", how="left")

    # Ajouter la colonne 'in_train' (si nécessaire)
    repaired_df["in_train"] = df["in_train"].values

    # Recalculer la colonne +40ans si nécessaire (assurer que cela soit cohérent avec l'attribut protégé)
    repaired_df["+40ans"] = (repaired_df["Patient Age"] >= 40).astype(int)

    # Sauvegarder le DataFrame final dans un fichier CSV
    repaired_df.to_csv(data_dir + "lfr_metadata.csv", index=False)

    return repaired_df


truc = apply_lfr(train_df)

# Tu peux également utiliser ce code pour appliquer LFR sur les données de test si nécessaire.
truc2 = apply_lfr(df=test_df)
repaired_df = pd.concat([truc, truc2], ignore_index=True)
repaired_df.to_csv(data_dir+"lfr_metadata.csv", index=False)
# train_df contiendra le dataset transformé et fusionné avec les chemins d'images.


step: 0, loss: 1.079316573709697, L_x: 1003.9171574987739,  L_y: 0.7373039327137676,  L_z: 0.0016690229395463108
step: 500, loss: 54632415.71837373, L_x: 54632414077.987015,  L_y: 16.403867142424872,  L_z: 0.0
step: 1000, loss: 0.33454109556910594, L_x: 66.61772043445421,  L_y: 0.6875655089853168,  L_z: 0.19916682423612
step: 1500, loss: 0.2806247369927808, L_x: 62.350370456647035,  L_y: 0.6840227670163771,  L_z: 0.14987208983449601
step: 2000, loss: 0.2805987466466061, L_x: 62.344767232722674,  L_y: 0.684042497250987,  L_z: 0.14984972968878477
step: 2500, loss: 0.2800909321039298, L_x: 62.442437905270936,  L_y: 0.680317084833193,  L_z: 0.1496167857153396
step: 3000, loss: 0.22118874217092524, L_x: 132.46938167041634,  L_y: 0.7259977409597662,  L_z: 0.0161195864045323
step: 3500, loss: 0.20962678664646317, L_x: 118.07814174472396,  L_y: 0.6879522872435335,  L_z: 0.022753416177385855
step: 4000, loss: 0.2092685753143676, L_x: 118.22911474255791,  L_y: 0.6870985257595308,  L_z: 0.0223296

In [33]:
repaired_train_df = apply_disparate_impact_remover(train_df)
repaired_test_df =  apply_disparate_impact_remover(test_df)

repaired_df = pd.concat([repaired_train_df, repaired_test_df], ignore_index=True)

repaired_df.to_csv(data_dir+"lfr_metadata.csv", index=False)


In [34]:
train_and_predict("lfr_metadata.csv", "lfr_preds.csv")

Les prédiction existent déjà à ./expe_log/lfr_preds.csv -- abandon de l'entraînement


In [35]:
lfr_pred = pd.read_csv("./expe_log/lfr_preds.csv")
lfr_pred = convert_to_all_numerical(lfr_pred)
metrics_after_lfr = getMetric(lfr_pred, sensitive_attr)

compare_to_base_preds(metrics_after_lfr)


0.4573 ---- base rate ---> 0.4573, (diff = 0.0000)
-0.2070 ---- SPD ---> -0.0077, (diff = 0.1993)
0.5608 ---- DI ---> 0.9832, (diff = 0.4224)
-0.1404 ---- equal_opportunity_difference ---> 0.0026, (diff = 0.1430)
-0.1347 ---- average_odds_difference ---> -0.0157, (diff = 0.1189)
-0.0595 ---- conditional_demographic_disparity ---> 0.0007, (diff = 0.0603)
0.5767 ---- smoothed_edf ---> 0.0189, (diff = -0.5577)
0.2311 ---- df_bias_amplification ---> -0.0267, (diff = -0.2578)


In [37]:
add_error_rate(lfr_pred, "Lfr")
error_rate_df

/tmp/ipykernel_85221/3799069778.py:20: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_85221/3799069778.py:25: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,method,global,+40ans,-40ans,M,F
0,Normal,23.800000,25.255624,21.072797,24.251070,23.404255
1,Rewight,27.400000,28.834356,24.712644,26.818830,27.909887
2,Dir,30.000000,NaN,30.000000,28.815977,31.038798
3,Lfr,26.333333,NaN,26.333333,25.106990,27.409262
4,Lfr,26.333333,NaN,26.333333,25.106990,27.409262


In [ ]:

plot_confusion_matrix_by_group(lfr_pred["labels"], lfr_pred["preds"], lfr_pred, group_columns=["+40ans"], labels=[0, 1])

## Post processing

In [ ]:
df=pd.read_csv("./expe_log/reweighted_preds.csv")
df.columns


In [ ]:
from aif360.algorithms.postprocessing.reject_option_classification import RejectOptionClassification


metric_name = "Statistical parity difference"
metric_ub = 0.5
metric_lb = -0.5




def apply_ROC_to_preds(test_df, weights=False):
    ROC = RejectOptionClassification(
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
        low_class_thresh=0.0001,
        high_class_thresh=0.999,
        num_class_thresh=100,
        num_ROC_margin=50,
        metric_name=metric_name,
        metric_ub=metric_ub,
        metric_lb=metric_lb
    )

    test_dt = test_dataset.copy()
    test_dt.scores = test_df["logits_1"].values.reshape(-1, 1)
    test_dt.labels = test_df["preds"]

    ROC = ROC.fit(test_dataset, test_dt)
    df_roc_val_pred = ROC.predict(test_dt)
    post_proc_metrics=get_group_metrics(
        y_true=test_dataset.labels[:,0],
        y_pred=df_roc_val_pred .labels[:,0],
        prot_attr=test_dt.protected_attributes[:, 0],
        pos_label=1,
        sample_weight=test_dt.features[:, test_dt.feature_names.index('WEIGHTS')] if weights else None ,
    )
    return post_proc_metrics


In [ ]:
apply_ROC_to_preds(test_df_reweighing, weights=True)
apply_ROC_to_preds(test_df_dir)
apply_ROC_to_preds(test_df_lfr)

In [ ]:
def cross_validate_ROC(test_df, metrics_to_try=None, class_thresholds=None, fairness_bounds=None, weights=False):
    test_dataset = BinaryLabelDataset(
        favorable_label=0,  
        unfavorable_label=1,  
        df=convert_to_all_numerical(test_df).select_dtypes(include=['int64', 'float64']),
        label_names=["labels"],
        protected_attribute_names=[protected_attribute]
    )

    # Default parameters
    if metrics_to_try is None:
        metrics_to_try = [
            "Statistical parity difference",
            "Equal opportunity difference",
            "Average odds difference"
        ]
    if class_thresholds is None:
        class_thresholds = [(0.01, 0.99), (0.001, 0.999)] 
    if fairness_bounds is None:
        fairness_bounds = [
            (-0.01, 0.01),
            (-0.05, 0.05),
            (-0.1, 0.1),
            (-0.25, 0.25)
        ]

  
    test_with_preds = test_dataset.copy(deepcopy=True)
    test_with_preds.labels = test_df["preds"].values.reshape(-1, 1)
    test_with_preds.scores = test_df["logits_1"].values.reshape(-1, 1)
    

    results_dfs = {}
    for metric in metrics_to_try:
        metric_results = []
        for low_thresh, high_thresh in class_thresholds:
            for lb, ub in fairness_bounds:
                try:
                   
                    # Initialize ROC with current parameters
                    ROC = RejectOptionClassification(
                        unprivileged_groups=unprivileged_groups,
                        privileged_groups=privileged_groups,
                        low_class_thresh=low_thresh,
                        high_class_thresh=high_thresh,
                        num_class_thresh=100,
                        num_ROC_margin=50,
                        metric_name=metric,
                        metric_ub=ub,
                        metric_lb=lb
                    )

                    
                    
                    # Fit ROC on both datasets (original and with predictions)
                    
                    ROC = ROC.fit(test_dataset, test_with_preds)
                    
                    print("Optimal classification threshold (with fairness constraints) = %.4f" % ROC.classification_threshold)
                    print("Optimal ROC margin = %.4f" % ROC.ROC_margin)
                    
                    # Apply the transformation to get fair predictions
                    transformed_dataset = ROC.predict(test_with_preds)
                    
                    # Calculate metrics using the transformed predictions
                    weight_column = None
                    if weights:
                        if 'WEIGHTS' in test_dataset.feature_names:
                            weight_column = test_dataset.features[:, test_dataset.feature_names.index('WEIGHTS')]
                    
                    metrics = get_group_metrics(
                        y_true=test_dataset.labels[:,0],
                        y_pred=transformed_dataset.labels[:,0],
                        prot_attr=test_dataset.protected_attributes[:, 0],
                        pos_label=1,
                        sample_weight=weight_column
                    )
                    
                    distance = check_distance_to_ideal(metrics)
                    result_row = {
                        'low_class_thresh': low_thresh,
                        'high_class_thresh': high_thresh,
                        'metric_lb': lb,
                        'metric_ub': ub,
                        'distance': distance,
                        **metrics
                    }
                    metric_results.append(result_row)
                except Exception as e:
                    print(e)
                    result_row = {
                        'low_class_thresh': low_thresh,
                        'high_class_thresh': high_thresh,
                        'metric_lb': lb,
                        'metric_ub': ub,
                        'distance': np.nan,
                        'error': str(e)
                    }
                    metric_results.append(result_row)
        
        df = pd.DataFrame(metric_results)
        df.set_index(['low_class_thresh', 'high_class_thresh', 'metric_lb', 'metric_ub'], inplace=True)
        results_dfs[metric] = df
    
    return results_dfs
results_dfs = cross_validate_ROC(test_df_dir)

In [ ]:
results_dfs["Statistical parity difference"]
results_dfs["Equal opportunity difference"]
results_dfs["Average odds difference"]

In [ ]:
from aif360.algorithms.postprocessing.calibrated_eq_odds_postprocessing import CalibratedEqOddsPostprocessing



def apply_CEO(test_df):

    cost_constraint = "fnr" # "fnr", "fpr", "weighted"
    cpp = CalibratedEqOddsPostprocessing(privileged_groups = privileged_groups,
                                        unprivileged_groups = unprivileged_groups,
                                        cost_constraint=cost_constraint,
                                        seed=42)
    
    pred_dataset = test_dataset.copy(deepcopy=True)
    pred_dataset.labels = test_df["preds"].values.reshape(-1, 1)
    pred_dataset.scores = test_df["logits_1"].values.reshape(-1, 1)
   

    cpp = cpp.fit(test_dataset, pred_dataset)
    df_ceqodds_val_pred = cpp.predict(pred_dataset)
    
    m=get_group_metrics(
        y_true=test_dataset.labels,
        y_pred=df_ceqodds_val_pred.labels[:,0],
        prot_attr=test_dataset.protected_attributes[:, 0],
        priv_group=1,
        pos_label=1,
    )
    return m

In [ ]:
apply_CEO(test_df_reweighing)
apply_CEO(test_df_dir)
apply_CEO(test_df_lfr)

## Conclusion